### Image input

In [2]:
from ipywidgets import FileUpload
from IPython.display import display
from langchain.agents import create_agent
from dotenv import load_dotenv
from langchain.messages import HumanMessage, SystemMessage

In [3]:
load_dotenv()

True

In [4]:
agent = create_agent(
    model="google_genai:gemini-3.6-flash"
)

In [5]:
uploader = FileUpload(accept = '.png', multiple = False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [6]:
print(uploader.value)

({'name': 'ChatGPT Image Aug 3, 2026, 06_11_59 PM.png', 'type': 'image/png', 'size': 1365418, 'content': <memory at 0x000001839BD4BE80>, 'last_modified': datetime.datetime(2026, 8, 3, 12, 41, 59, 376000, tzinfo=datetime.timezone.utc)},)


In [7]:
import base64
#getting the first uploded file dict
uploader_file = uploader.value[0]

#this is a memory view
content_mv = uploader_file['content']

#convert memoryview -> bytes
img_bytes = bytes(content_mv)

#now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")


In [8]:
multimodal_question = HumanMessage(content = [
    {"type": "text","text":"Tell me about the color of the shirt and pant in this image"},
    {"type":"image","base64": img_b64, "mime_type":"image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

In [9]:
print(response['messages'][-1].content[0]['text'])

Based on the image:

* **Shirt:** 
  * The outer button-up shirt is a **deep chocolate brown**. 
  * The inner top underneath it is **black**.
* **Pants:** The wide-leg pants/jeans are **black** (or a slightly washed dark charcoal black).


### Audio input

In [10]:
import sounddevice as sd 
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

In [11]:
duration = 10
sample_rate = 44100

In [20]:
print("Recording....")
audio = sd.rec(int(duration * sample_rate), samplerate = sample_rate, channels=1)

for _ in tqdm(range(duration * 10)):
    time.sleep(0.1)
sd.wait()
print("Done")

buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording....


100%|██████████| 100/100 [00:10<00:00,  9.82it/s]

Done


In [21]:
agent = create_agent(
    model = "google_genai:gemini-3.6-flash"
)

multimodal_question = HumanMessage(content=[
    {"type":"text", "text":"Complete the task said in the audio file"},
    {"type":"audio","base64":aud64,"mime_type":"audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

In [22]:
print(response['messages'][-1].content[0]['text'])

To add fractions with the same denominator, add the numerators and keep the denominator the same:

$$\frac{3}{5} + \frac{1}{5} = \frac{3 + 1}{5} = \frac{4}{5}$$
